# Catálogo do dataset Argus

## Escopo da exploração

Este notebook explora o dump MySQL local do Argus. Ele não consulta BigQuery: o dump é uma fotografia local e pode diferir da fonte analítica em cobertura e atualização.

Os nomes de tabela aqui são físicos. Em geral:
- tabelas com prefixo `signal_` guardam sinais e metadados de sinais;
- `collector_run` e tabelas parecidas descrevem execuções dos coletores;
- `incidents*` e `incident_label` tratam de tickets e classificação de incidentes;
- `live_*`, `player_*`, `device_*`, `geo_*` e `error_cat_*` concentram experiência e QoE;
- `prom_*`, `rpaas_*`, `ca_*` e `tsuru_event` representam infraestrutura e operação;
- `dashboard_artifact` e `field_metrics` guardam artefatos e métricas auxiliares.

In [ ]:
from dump_explorer.catalog import (
    EXPECTED_TABLES,
    compare_expected_tables,
    load_column_catalog,
    load_table_catalog,
)
from dump_explorer.charts import volume_bar
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine

settings = load_settings()
engine = create_db_engine(settings)

## Catálogos disponíveis

As estimativas abaixo vêm dos metadados do MySQL. `table_catalog` mostra o nome da tabela e o volume estimado; `column_catalog` mostra as colunas, os tipos e as chaves indexadas. Esses dois catálogos servem para descobrir o que existe no dump sem executar `COUNT(*)` pesado em cada tabela.

In [ ]:
table_catalog = load_table_catalog(engine)
column_catalog = load_column_catalog(engine)

display(table_catalog)
display(column_catalog)

## Cobertura documentada

Compare o dump com a lista esperada pelo laboratório. O status pode ser `present`, `missing` ou `undocumented`:
- `present`: a tabela existe no dump e também está documentada;
- `missing`: a tabela era esperada, mas não apareceu no dump;
- `undocumented`: a tabela existe no dump, mas não faz parte da lista esperada.

In [ ]:
table_coverage = compare_expected_tables(
    actual=table_catalog["table_name"].tolist(),
    expected=EXPECTED_TABLES,
)
display(table_coverage)

## Volume por tabela

O gráfico está marcado como estimativa do MySQL. Contagens exatas não são computadas para tabelas grandes, para preservar o dump e evitar consultas caras. Em outras palavras: a altura da barra mostra o volume aproximado disponível no catálogo, não um `COUNT(*)` completo do banco inteiro.

In [ ]:
volume_bar(table_catalog).show()